# Linear Regression via Gradient Descent — From Scratch

Predicts bone mineral density (`bmd`) using batch gradient descent implemented from scratch
(no `sklearn.linear_model`), tracking how each weight evolves over training and evaluating with
MSE, RMSE, and MAE.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

## Load & Prepare Data

In [ ]:
data = pd.read_csv("./data/bmd.csv", sep=",", engine="python")
data.drop("id", axis=1, inplace=True)

data = pd.get_dummies(data, drop_first=True)
data = data.fillna(method="ffill").fillna(method="bfill")

In [ ]:
X = data.drop("bmd", axis=1)
y = data["bmd"]

feature_names = ["bias"] + list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Gradient Descent

In [ ]:
X_train_b = np.c_[np.ones((X_train.shape[0], 1)), X_train]
X_test_b = np.c_[np.ones((X_test.shape[0], 1)), X_test]

learning_rate = 0.003
n_iterations = 1000
n_features = X_train_b.shape[1]
weights = np.zeros(n_features)
weights_history = []

In [ ]:
for i in range(n_iterations):
    gradients = 2 / X_train_b.shape[0] * X_train_b.T.dot(X_train_b.dot(weights) - y_train)
    weights -= learning_rate * gradients
    weights_history.append(weights.copy())

weights_history = np.array(weights_history)

In [ ]:
plt.figure(figsize=(12, 6))
for i in range(n_features):
    plt.plot(weights_history[:, i], label=f"Weight {i}")
plt.xlabel("Iteration")
plt.ylabel("Weight Value")
plt.title("Weight Changes During Training (Gradient Descent)")
plt.legend()
plt.show()

## Evaluation

In [ ]:
y_pred = X_test_b.dot(weights)

mse = np.mean((y_test - y_pred) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_test - y_pred))

print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)

In [ ]:
weights_df = pd.DataFrame({"Feature": feature_names, "Weight": weights})
print("\nFinal Feature Weights:")
print(weights_df)